In [ ]:
####################### N O  T O C A R ############################################
%load_ext autoreload
%autoreload 2
import sys
import os

# Agrega la ruta raíz del proyecto si no está
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# import Configs.configuracion_general as config
import services.Carga.cargar_datos_csv as carga 
from services.Utils.utilidades import *
from services.Correctores.corrector_utils import *
from configs.manager_diccionario_variables import * 
from services.Graficado.graficar_series_y_guardar import graficar_series_y_guardar
from services.Graficado.graficar_mapa_de_posiciones import graficar_mapa_de_posiciones
from services.Graficado.graficar_mapa_de_despliegue import graficar_mapa_de_despliegue


##################################################################################

In [ ]:
rutas_de_sondas, seriales_encontrados = carga.buscar_nombre_de_archivo_de_sonda()
rutas_de_sondas

In [ ]:
diccionario_de_datos_de_sondas = carga.cargar_datos_de_sonda(rutas_de_sondas, seriales_encontrados)

In [ ]:
diccionario_de_datos_de_sondas.keys()

In [ ]:
# para 10.3
diccionario_de_sondas_en_fechas = carga.seleccionar_rango_de_fechas(diccionario = diccionario_de_datos_de_sondas, buscar_fechas_anteriores_al_estudio = False)
datos_de_sondas_sin_duplicados = carga.buscar_y_eliminar_duplicados(diccionario_de_sondas_en_fechas)
datos_ordenados = carga.ordernar_datos_por_fecha(datos_de_sondas_sin_duplicados)

In [ ]:
datos_ordenados.keys()

In [ ]:
for serial in seriales_encontrados:
    if serial in datos_ordenados:
        datos_ordenados[serial]["tspan_rounded"] = datos_ordenados[serial]["tspan_de_envio"]


In [ ]:
# datos_ordenados["4857577"]

In [ ]:
# Eliminar datos espurios (solo se revisa si hay valores de rapidez superiores a 2 m/s y se elimina toda la fila)
datos_finales = eliminar_datos_espurios(datos_ordenados)

In [ ]:
# Agregar componentes de la velocidad al diccionario con los dataframe de cada sonda
datos_finales = carga.agregar_componentes_de_la_velocidad(datos_finales)

In [ ]:
tabla_de_porcentajes = calcular_porcentaje_de_datos_recibidos(datos_finales)

In [ ]:
# datos_finales

In [ ]:

tabla_de_porcentajes

In [ ]:
graficar_series_y_guardar(datos= datos_finales, mostrar_figura=False)



In [ ]:
graficar_mapa_de_posiciones(datos=datos_finales, mostrar_figura=False)

In [ ]:
graficar_mapa_de_despliegue(mostrar_figura=False)

In [ ]:
# ruta_a_carpeta = crear_ruta_a_carpeta(get_carpeta_guardado_datos_procesados())
# ruta_pickle = os.path.join(ruta_a_carpeta, get_nombre_archivo_datos_procesados())
# diccionario = cargar_diccionario_pickle(ruta_pickle)

# Guardar datos del estudio en archivo .pkl
carpeta_de_destino = crear_ruta_a_carpeta(get_carpeta_guardado_datos_procesados())
nombre_de_archivo = get_nombre_archivo_datos_procesados()
guardar_diccionario_como_pickle(diccionario = datos_finales, 
                                ruta = carpeta_de_destino, 
                                nombre_archivo=nombre_de_archivo)

# Guardar Tabla de porcentajes
ruta_a_carpeta_de_datos_procesados = crear_ruta_a_carpeta(get_carpeta_guardado_datos_procesados())
guardar_porcentajes_en_excel(data= tabla_de_porcentajes, ruta= ruta_a_carpeta_de_datos_procesados, nombre_de_archivo=get_nombre_del_excel_de_porcentajes())